In [ ]:
import coffea
from coffea.nanoevents.methods import vector
import coffea.hist as hist
import uproot
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
import sys,os

# from pyjet import cluster
import awkward as ak

from sklearn.cluster import DBSCAN

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema,BaseSchema

sys.path.append(os.getcwd().replace('scripts', 'lib'))
import util

# from numba import jit

# Open data files

 * First file contains collision data taken with large MET in the trigger in 2016. This is a sub-set of the analysis nTuple that you will use in the rest of the exercise.
 * The 2nd file contains signal simulation with $H\rightarrow SS \rightarrow 4b$ events

In [ ]:
fpath = "/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/data/displacedJetMuon_ntupler.root"
data = NanoEventsFactory.from_root(fpath,
                                   schemaclass=BaseSchema,
                                   treepath="ntuples/llp"
                                  ).events()

fpath ="/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/displacedJetMuon_ntupler.root"

signal = NanoEventsFactory.from_root(fpath,
                                   schemaclass=BaseSchema,
                                   treepath="ntuples/llp"
                                  ).events()

## The tree is loaded with the "coffea" analysis framework
The events are loaded into an "awkward arrays", i.e. arrays with irregular sizes
Our event-based data fits into awkward arrays very naturally!
 - Tree:
 -  |-muonPt :[ [Pt1,Pt2], [Pt1] ...]
 -  |-muonEta:[ [Eta1,Eta2], [Eta1] ...]

In [ ]:
# this awkward array has 16965 events, of the type "event"
data

In [ ]:
# shows the branches of the tree
data.fields

## Accessing a branch like this will give the data in the format of (awkward)arrays
 - Note that the type becomes "int32"
 - For event-level quantity (number of total rechits),
   the length of array is the **same** as the number of events

In [ ]:
data.ncscRechits

 - For object-level quantity (X pos. of each rechit), the first dimension of array is the same as the number of events, but the 2nd dimension of the array is **variable (var, float32)** 

In [ ]:
data.cscRechitsX

## Pack all the rechit properties arrays in 1 object

In [ ]:
def pack(events,obj_str):
    obj  = ak.zip(
                {k.replace(obj_str,""):getattr(events,k) for k in events.fields if k.startswith(obj_str)}
                ,with_name="PtEtaPhiMLorentzVector",
                behavior=vector.behavior
               )
    return obj

In [ ]:
rechits = pack(data,"cscRechits")

In [ ]:
d_rechits = rechits
s_rechits = pack(signal,"cscRechits")

 - These properties mostly follow from the CMSSW definition: [here](https://github.com/cms-sw/cmssw/blob/master/DataFormats/CSCRecHit/interface/CSCRecHit2D.h)
 - Our nTuplizer only dropped a few properties and computed a few new ones
 - We will use the following properties:
   - Eta,Phi
   - X,Y,Z
   - Tpeak (Time calculated from strip input), Twire (Time calculated from wire input) [src code](https://github.com/cms-sw/cmssw/blob/master/RecoLocalMuon/CSCRecHitD/src/CSCMake2DRecHit.cc)
   - Station

**Discussion 3.1 (answers).** Question 1, verbatim: "Can you think of some of the properties of a rechit?" Question 2, verbatim: "Are there disadvantage/limitation for using rechit as the inputs of MDS?"

Q1: The properties most relevant for MDS are the position (x, y, z, eta, phi) and the time (Tpeak from the strips, Twire from the wires). The station is also useful, as are the chamber and ring that identify where the rechit sits.

Q2: Rechit reconstruction is designed for a single muon, so it can miss details of how a shower develops. A dedicated machine-learning reconstruction might recover those details. Multiplicity is the most important MDS feature, and rechits keep it, so the limitation is small.


In [ ]:
rechits.fields

In [ ]:
# Check number of rechits per event
ak.num(rechits)

## Look at the 2D event display of rechits

In [ ]:
def displayRZ(i_event,rechits=[],cls=[]):
    
    #Select the rechits in the i-th event
    i_rh = rechits[i_event] if len(rechits)>0 else None
    i_cls = cls[i_event] if len(cls)>0 else None
    
    #Make a figure with 2 subplots
    fig, axs = plt.subplots(1,2, figsize=(18,8))

    axs = axs.flatten()
    #plot clusters:
    if len(cls)>0:
        #Eta-phi plot:        
        s1=axs[0].scatter(i_cls.Eta,i_cls.Phi,s=i_cls.Nhit,label="Cluster")
        for cl in i_cls:
            cone = plt.Circle((cl.Eta, cl.Phi), 0.4, color='b', fill=False,)
            axs[0].add_patch(cone)        
        
        #R-Z plot:        
        cls_r = (i_cls.X**2+i_cls.Y**2)**0.5        
        axs[1].scatter(np.abs(i_cls.Z),cls_r,s=i_cls.Nhit)            
    
    #plot rechits:
    if len(rechits)>0:
        #Eta-phi plot:
        s2=axs[0].scatter(i_rh.Eta,i_rh.Phi,s=1,label="Rechits")
        rh_r = (i_rh.X**2+i_rh.Y**2)**0.5    
        
        #R-Z plot:        
        axs[1].scatter(np.abs(i_rh.Z),rh_r,s=1)

    #label eta-phi plot
    axs[0].text(0.1,0.9,"%s-th event"%i_event,transform=axs[0].transAxes)
    axs[0].set_ylim(-np.pi,np.pi)
    axs[0].set_xlim(-5,5)
    axs[0].set_ylabel(r"$\phi$")
    axs[0].set_xlabel(r"$\eta$")
    
    #label RZ plot
    MB_xmin = 300
    util.drawRZ(axs[1],MB_xmin)
    axs[1].set_xlim(MB_xmin,None)
    axs[1].set_xlabel("|z|[cm]")
    axs[1].set_ylabel("R[cm]")
    
    #make legend
    handles, labels = [(a + b) for a, b in zip(axs[0].get_legend_handles_labels(), 
                                           axs[1].get_legend_handles_labels())]
    leg=fig.legend(handles, labels,
                   bbox_to_anchor=(1.0, 1.0), loc='upper left')   
    for ax in axs:
        hep.cms.label(ax=ax,data=True,label="Preliminary")    
    for l in leg.legendHandles:
        l._sizes = [30]
    plt.tight_layout()
    return

### Try to look at the rechits from different events!
Hint: find an event that has a large number of rechits first to get an interesting event!

In [ ]:
# check which events have large number of rechits
np.where(ak.num(rechits)>400)

In [ ]:
plt.style.use(hep.style.CMS) 
displayRZ(1,rechits)

# Clustering the rechits

### First read the discussion in the lesson: [link](https://kakwok.github.io/MDS_CMSDAS/03-MDS_reco/index.html#clustering-algorithm)

### Complete the function `computeCluster`, `computeStationProp` and `computeME11_12`

First read the definition of the cluster properties: [here](https://kakwok.github.io/MDS_CMSDAS/03-MDS_reco/index.html#cluster-properties)

Then try to implement them in the relevant functions.

In particular, 

### $\mathrm{avgStation10} = \frac{\sum_i (i \times nhits_i)}{\sum_i nhits_i}$

where `i` sums over the stations with at least 10 hits.

We are using DBSCAN algorithm, implemented with the python lib: `sklearn.cluster`

In [ ]:
def DBScan(rechits,nCore,eps):
    points=ak.zip([rechits.Eta,rechits.Phi],highlevel=False)

    all_clusters = []
    for x,rechit in zip(points,rechits):
        n_clusters_=0
        core_samples_mask=[]
        ## format into DBSCAN input format x=[ [x1,y1],[x2,y2] ... ]
        x=np.stack(ak.unzip(x),axis=1)
        i_clusters = []
        if len(x)>0:
            db = DBSCAN(eps=eps, min_samples=nCore).fit(x)
            core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
            core_samples_mask[db.core_sample_indices_] = True
            labels = db.labels_
            n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)

            ## Loop over clusters in this event
            for iCs in range(n_clusters_):

                #Compute each cluster's properties:
                #    Input = rechits that belongs to this cluster (done via array masking)
                cls = computeCluster(rechit[labels==iCs])
                i_clusters.append(cls)
        all_clusters.append(i_clusters)
    return ak.Array(all_clusters)

# This function compute the cluster properties.
# The input `rechits` is an *array* of rechits used in the cluster
# This function is ran for each event
def computeCluster(rechits):
    cls = {}

    cls["Nhit"] = len(rechits)
    # position: average of the constituent rechit positions
    cls["Eta"] = np.mean(ak.to_numpy(rechits.Eta))
    cls["Phi"] = np.mean(ak.to_numpy(rechits.Phi))
    cls["X"] = np.mean(ak.to_numpy(rechits.X))
    cls["Y"] = np.mean(ak.to_numpy(rechits.Y))
    cls["Z"] = np.mean(ak.to_numpy(rechits.Z))
    # time: average of the wire time and the strip time of each constituent rechit
    cls["Time"] = np.mean(ak.to_numpy((rechits.Twire + rechits.Tpeak)/2.0))

    avgStation10,nStation10 = computeStationProp(rechits)
    nME11_12 = computeME11_12(rechits)

    cls["nStation10"] = nStation10
    cls['avgStation10'] = avgStation10
    cls["nME11_12"] = nME11_12

    return cls

# This function compute the cluster properties related to cluster stations
# The input `rechits` is an *array* of rechits used in the cluster in each event
# This function is ran for each event
def computeStationProp(rechits):
    # station can range from -4 to 4
    uniqueStations = np.array([-4,-3,-2,-1,1,2,3,4])
    nStation10   = 0
    avgStation10 = 0
    TotHitsStation10 = 0
    for i_station in uniqueStations:

        # total hits in the i-th station e.g. -4
        nRechit_in_i_station = sum(rechits.Station==i_station)

        #
        if nRechit_in_i_station >=10:
            # number of stations with >=10 hit
            nStation10   += 1
            # average of stations position with >=10 hit, weighted by the number of rechits in each station
            avgStation10 += i_station*nRechit_in_i_station

            # total hits in station with at least 10 hits (i.e. sum of weights of avgStation10)
            TotHitsStation10 += nRechit_in_i_station

    # guard: a cluster with no station having >=10 hits would divide by zero
    avgStation10 = avgStation10/TotHitsStation10 if TotHitsStation10 > 0 else -999.0
    return avgStation10,nStation10

# This function compute the total number of rechits in ME11 and ME12
# The input `rechits` is an *array* of rechits used in the cluster
# This function is ran for each event
def computeME11_12(rechits):
    # ME11 = station 1 ring 1, ME12 = station 1 ring 2
    # NOTE: the 'Ring' field name is assumed; confirm with rechits.fields (cell 15)
    is_ME1 = np.abs(rechits.Station) == 1
    is_ring12 = (rechits.Ring == 1) | (rechits.Ring == 2)
    return int(ak.sum(is_ME1 & is_ring12))


**Explanation.** Each cluster property is a direct transcription of the definitions in the lesson:
- Nhit = number of constituent rechits.
- Eta, Phi, X, Y, Z = plain averages of the rechit positions.
- Time = average of (Twire + Tpeak)/2 over the rechits.
- nStation10 = number of stations with at least 10 rechits.
- avgStation10 = sum(i * nhits_i)/sum(nhits_i) over the stations with at least 10 rechits.
- nME11_12 = number of rechits in station 1 with ring 1 or 2.

Pitfalls: (1) Phi is a naive average and does not handle the wrap at +/-pi, which is fine for the clusters here but worth knowing. (2) avgStation10 divides by zero if no station has 10 rechits; a guard now returns -999 for that case, which the histograms drop as out of range. (3) The ring field name is an assumption; confirm it in rechits.fields.

In [ ]:
%%time

minPoint = 50
dR = 0.2
cls = DBScan(rechits,minPoint,dR)

In [ ]:
s_cls = DBScan(s_rechits,minPoint,dR)


**Challenge 2 (explanation).** The signal rechits (s_rechits, built in cell 13) are clustered with the same DBSCAN parameters (minPts = 50, dR = 0.2) and stored in s_cls, so the signal and data clusters go through identical code.


In [ ]:
cls.fields

## Try to see if the cluster in the previous event makes sense!
Hint: find an event that has a reconstructed cluster first

Do you see a cluster in the event that you chose?

In [ ]:
# check which events have at least 1 cluster
np.where(ak.num(cls)>=1)

In [ ]:
displayRZ(1,rechits,cls)

# Examine cluster properties

In [ ]:
samples=[
    {"cls":cls,"label":"data"},

    # you may add more lines like this one to overlay them on plots
    # e.g. {"cls":s_cls,"label":"signal"},
    # where `s_cls` is the clusters from signals
    {"cls":s_cls,"label":"signal"},
]
density=False


**Challenge 2 (explanation, plots).** Adding the signal line to `samples` overlays signal (s_cls) and data (cls) in every plot below. Compare N_rechits, Time, and nME11_12 in particular: these are the variables that separate signal from background in the next discussion. Reading the plotting code, each cell fills a Hist with a `sample` category axis and draws the normalised or raw distributions side by side.


## Number of clusters and N_rechits

In [ ]:
plt.style.use(hep.style.CMS)  
fig, axs = plt.subplots(1,2,figsize=(16,8))

axs = axs.flatten()
h1= hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("nClusters", "nClusters", 10, 0, 10))
h2= hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("Nhit", r"$N_{rechit}$", 20, 0, 1000))

for sample in samples:
    cls = sample['cls']
    label = sample['label']
    h1.fill(sample=label ,nClusters=ak.num(cls))
    h2.fill(sample=label ,Nhit = ak.flatten(cls.Nhit))

hist.plot1d(h1,density=density,ax=axs[0])    
hist.plot1d(h2,density=density,ax=axs[1])
for ax in axs:
    
    hep.cms.label(ax=ax,data=True,label="Preliminary")        
    if density: ax.set_ylabel("Density")
plt.tight_layout()


# Eta, Phi distributions

In [ ]:
plt.style.use(hep.style.CMS)  
fig, axs = plt.subplots(1,2,figsize=(16,8))

axs = axs.flatten()
h1= hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("eta", r"$\eta$", 40, -4, 4))
h2= hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("phi", "phi", 64, -3.2, 3.2))

for sample in samples:
    cls = sample['cls']
    label = sample['label']
    h1.fill(sample=label ,eta = ak.flatten(cls.Eta))
    h2.fill(sample=label ,phi = ak.flatten(cls.Phi))

hist.plot1d(h1,density=density,ax=axs[0])    
hist.plot1d(h2,density=density,ax=axs[1])
for ax in axs:
    
    hep.cms.label(ax=ax,data=True,label="Preliminary")        
    if density: ax.set_ylabel("Density")
plt.tight_layout()


# R, Z distributions

In [ ]:
plt.style.use(hep.style.CMS)  
fig, axs = plt.subplots(1,2,figsize=(16,8))

axs = axs.flatten()
h2 = hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("R", "R", 100, 0, 1000))        
h3 = hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("Z", "Z", 120, -1200, 1200))            

for sample in samples:
    cls = sample['cls']
    label = sample['label']
    h2.fill(sample=label ,R = ak.flatten((cls.Y**2+cls.X**2)**0.5))
    h3.fill(sample=label ,Z = ak.flatten(cls.Z))        

hist.plot1d(h2,density=density,ax=axs[0])
hist.plot1d(h3,density=density,ax=axs[1])
for ax in axs:
    
    hep.cms.label(ax=ax,data=True,label="Preliminary")        
    if density: ax.set_ylabel("Density")
plt.tight_layout()


# Time, nStation10, AvgStation10 distributions

In [ ]:
plt.style.use(hep.style.CMS)  
fig, axs = plt.subplots(1,3,figsize=(24,8))

axs = axs.flatten()
h1= hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("Time", 'ClusterTime[ns]',50, -100, 100))                        
h2= hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("nStation10", "Nstation10", 8, 0, 8))    
h3= hist.Hist("Events",hist.Cat("sample","sample"),hist.Bin("avgStation10", "AvgStation10", 40, -4, 4))             

for sample in samples:
    cls = sample['cls']
    label = sample['label']
    h1.fill(sample=label ,Time = ak.flatten(cls.Time))
    h2.fill(sample=label ,nStation10 = ak.flatten(cls.nStation10))
    h3.fill(sample=label ,avgStation10 = ak.flatten(cls.avgStation10))    
    

hist.plot1d(h1,density=density,ax=axs[0])    
hist.plot1d(h2,density=density,ax=axs[1])
hist.plot1d(h3,density=density,ax=axs[2])
for ax in axs:
    
    hep.cms.label(ax=ax,data=True,label="Preliminary")        
    if density: ax.set_ylabel("Density")
plt.tight_layout()


**Discussion 3.2 (answer).** Question, verbatim: "Which variables can be used to distinguish between signal and background?"

The distributions of N_rechits, Time, and ME11_12 differ strongly between signal and background. Signal clusters are made of many rechits, have a Time that is spread in a distinct way, and carry more ME1/1 and ME1/2 rechits. Cuts on these variables separate signal from background. nStation10 and Eta are weaker on their own, which is why the bonus cell combines nStation10 with eta.


# Bonus: Cluster ID

In the above plot, you should see there's no clear distinction between signal and data in `nStation10` and `Eta`, but some combination of nStation10 and eta may also help us to distinguish between signal and background.

We will use these variable to design a "ClusterID" later.

In [ ]:
h= hist.Hist("Events",
              hist.Bin("nStation10", "Nstation10", 8, 0, 8),
              hist.Bin("eta", r"$\eta$", 40, -3, 3)
             )    
h.fill(nStation10=ak.flatten(cls.nStation10),
      eta = ak.flatten(cls.Eta))
h_group = h.group("nStation10",hist.Cat("nStationGrp","nStationGrp"),
                 {
                     "nStation=1":slice(0,2),
                     "nStation>=1":slice(2,None),                     
                 })

In [ ]:
hist.plot1d(h_group,density=True)

In [ ]:
h.to_hist().project("eta","nStation10").plot2d_full()

# MDS reconstruction efficiency for signal

When an LLP decay in CSC, we want to know 
 - *how often* it can make an MDS cluster (this is cluster efficiency) and
 - where does the LLP decay when this happens
 
In this part, we will make a plot of MDS efficiency as a function LLP decay position and try to understand it. 

# load a signal ntuple

In this ntuple, the DBSCAN algorithm is already performed to reconstruct clusters from rechits, so only clusters and the cluster properties that you've seen in the previous exercise are stored

In [ ]:
fpath ="/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/nocuts/ggH_HToSSTobbbb_MH-125_MS-40_ctau-1000_137000pb_weighted.root"

signal = NanoEventsFactory.from_root(fpath,
                                   schemaclass=BaseSchema,
                                   treepath="MuonSystem"
                                  ).events()

In [ ]:
signal

# Selections
Here we will select for events that have only one LLP decaying in the CSCs and calculate the probabilty of a cluster being reconstructed given that an LLP decayed in the CSC.
You will use the following branches:
* `gLLP_csc`: Boolean, a LLP-level variable that tells you whether an LLP decayed in CSC
* `gLLP_decay_vertex_z`: LLP-level variable that returns LLP lab frame decay vertex in Z [cm]
* `cscRechitCluster_match_gLLP_csc`: Boolean, cluster-level variable that returns true of the cluster is matched to an LLP ($\Delta R (cluster, LLP) < 0.4$) that decayed in CSC
* `cscRechitCluster_match_gLLP_decay_z`: float, cluster-level variable that returns the LLP decay vertex in z if its matched to an LLP

## Ex: Denominator selection

We will select events with only 1 LLP decaying in CSC.

There are 2 LLPs in each events. 

Compute the number of LLP that decays in the CSC first, then make a selection mask

How many events are there?

This will be the denominator of your signal efficiency.

In [ ]:
nLLPinCSC = ak.sum(signal.gLLP_csc,axis=1)

In [ ]:
denom_sel = (nLLPinCSC == 1) & signal.gLLP_csc


**Explanation.** nLLPinCSC is an event-level count, so (nLLPinCSC == 1) is one boolean per event. Multiplying by signal.gLLP_csc, which is one boolean per LLP, keeps only the LLPs that decayed in the CSC, in events where exactly one LLP did. Pitfall: an event-level array must not be used as the object mask on its own; the & broadcasts it onto the LLP level.

**Denominator answer.** The count of events with one LLP in the CSC is printed in cell 53. It is the denominator of the efficiency. Divide it by the total number of generated events to get the CSC acceptance.


In [ ]:
print("Events with 1 LLP decaying in CSC: ", ak.sum(denom_sel))
print("Total generated signal events ", len(signal))
print("Acceptance in CSC ", ak.sum(denom_sel)/len(signal))

In [ ]:
# z coordinates of the LLP in the denominator
# we sum over the LLPs in the event axis(axis=1) [which there is only 1 LLP]
llp_z = ak.sum(abs(signal.gLLP_decay_vertex_z[denom_sel]),axis=1)

## Ex: Numerator selection 

Select events with only 1 cluster AND in the denominator.

Among those events, select the ones that there's one cluster matched to the LLP.

In [ ]:
nClsInCSC = ak.sum(signal.cscRechitCluster_match_gLLP_csc,axis=1)

In [ ]:
numer_sel = (nClsInCSC == 1) & (nLLPinCSC == 1) & signal.cscRechitCluster_match_gLLP_csc


**Explanation.** Three conditions: exactly one cluster matched to a CSC LLP (event-level), exactly one LLP in the CSC (event-level), and the cluster itself is matched to a CSC LLP (object-level). The first two are broadcast onto the cluster level by the &. nClsInCSC counts clusters matched to a CSC LLP, as defined in cell 56.

In [ ]:
# z coordinates of the LLP in the numerator
# we sum over the clusters in the event axis(axis=1)[which there is only 1 cls]
cls_z = ak.sum(abs(signal.cscRechitCluster_match_gLLP_decay_z[numer_sel]),axis=1)

## EX: Inclusive efficiency

What is the overall efficiency of reconstructing an MDS cluster of 50 hits for this signal?

In [ ]:
print("Events with 1 LLP decaying in CSC: ", ak.sum(denom_sel))
print("Events with 1 LLP decaying in CSC and a cluster matched to LLP: ", ak.sum(numer_sel))
print("Inclusive efficiency = %.2f"%(ak.sum(numer_sel)/ak.sum(denom_sel)))


**Inclusive efficiency (answer).** The notebook question is "What is the overall efficiency of reconstructing an MDS cluster of 50 hits for this signal?"

The efficiency is the number of events with one CSC LLP and a matched cluster, divided by the number of events with one CSC LLP. Both counts are printed by cell 60. Because each selected event has exactly one LLP and at most one matched cluster, ak.sum counts events directly. I did not execute the notebook, so the printed value is not quoted here.


# Fill the histogram 

This is an **example** of filling a histogram with a `category` axis (for different samples), 

and a `Bin` axis for storing the values. 

This is a compact way of handling the histograms. 

You will use this type of histograms later on in the exercise. 

Which varaible should we fill-in in the function?

In [ ]:
#Create a 2D histogram with a "sample" axis and an axis
hz= hist.Hist("Events",
              hist.Cat("sample","sample"),
              hist.Bin("z", "LLP decay Z[cm]", 35, 400, 1075)
             )

#Fill the LLP decay position in CSC in the denominator bin
hz.fill(sample="denom", z = llp_z[llp_z>0])
#Fill the LLP decay position in CSC matched to a cluster in the numerator bin
hz.fill(sample="numer", z = cls_z[cls_z>0])


**Notebook question (answer).** Question, verbatim: "Which varaible should we fill-in in the function?"

For the denominator, fill the LLP decay Z of the events with one LLP in the CSC (llp_z from cell 54). For the numerator, fill the LLP decay Z of the matched clusters (cls_z from cell 58). Both are in cm. The filter `>0` drops the zero entries that appear for events outside the selection, which would otherwise land in the underflow bin.


# Efficiency v.s. decay position

We will plot the cluster reconstruction efficiency with repect to the LLP decay position and overlay the steel region in the plot.

Here's a function to draw the CSC steel region.

In [ ]:
## A function to draw the CSC steel region
import matplotlib.patches as patches

def drawCSCz(ax):
    ax.set_xlim(350,1150)
    (xmin,xmax) = ax.get_xlim()

    xmin = 350
    y_max = ax.get_ylim()[1]

    preME11 = patches.Rectangle((xmin, 0), 568-xmin, 2,color='grey',alpha=0.3)
    ME11_12 = patches.Rectangle((632, 0), 39, 2,color='grey',alpha=0.3)
    ME12_2  = patches.Rectangle((724, 0), 65, 2,color='grey',alpha=0.3)
    ME2_3   = patches.Rectangle((849, 0), 62, 2,color='grey',alpha=0.3)
    ME3_4   = patches.Rectangle((970, 0), 32, 2,color='grey',alpha=0.3)
    beyond  = patches.Rectangle((1075, 0),150, 2,color='grey',alpha=0.3)

    ax.text(570*1.045, y_max*1.02, 'ME1/1', fontsize=16,rotation=90,weight='bold')
    ax.text(670*1.02, y_max*1.02, 'ME1/2-3', fontsize=16,rotation=90,weight='bold')
    ax.text(800, y_max*1.02, 'ME2', fontsize=16,rotation=90,weight='bold')
    ax.text(920, y_max*1.02, 'ME3', fontsize=16,rotation=90,weight='bold')
    ax.text(1015, y_max*1.02,'ME4', fontsize=16,rotation=90,weight='bold')
    ax.text(xmin+15 ,y_max*0.5, "Steel", fontsize=26,rotation=90,weight='bold')
    ax.text(xmax-20,y_max*0.5, "Beyond CMS", fontsize=26,rotation=90,weight='bold')

    ax.add_patch(preME11)
    ax.add_patch(ME11_12)
    ax.add_patch(ME12_2)
    ax.add_patch(ME2_3)
    ax.add_patch(ME3_4)
    ax.add_patch(beyond)
    return ax

## Plotting the efficiency 

We can obtain the 1D histogram by integrating the `numer`/`denom` bin in the 2D histogram.

e.g. `hz.integrate("sample",'numer')` will return a 1D histogram with only the numerator content.

In [ ]:
plt.style.use(hep.style.CMS)  
fig, ax = plt.subplots(1,1,figsize=(12,9))

# We use the "clopper-pearson" 
# Have a read of what it's doing here (the "efficiency" option): 
# https://hist.readthedocs.io/en/latest/reference/hist.intervals.html#hist.intervals.ratio_uncertainty

hist.plotratio(num=hz.integrate("sample",'numer'),
               denom=hz.integrate("sample",'denom'),xerr=True,
               error_opts={"linestyle":'none',"lw":4},ax=ax,
               unc = "clopper-pearson"
               )

ax.set_ylabel("Cluster Efficiency")
ax.set_xlabel("LLP z decay position [cm]")

ax=drawCSCz(ax)

ax.set_ylim(0,1)
ax.set_xlim(350,1150)
ax.grid() 
hep.cms.label(ax=ax,com=13)
plt.tight_layout()

**Discussion 3.3 (answer, part 1 and 2).** Questions, verbatim: "Why does the efficiency drops off at the two ends of the muon detectors?" and "How does the efficiency varies between the muon stations? Do you understand why?"

The efficiency falls at the two ends for different reasons. At large z, beyond the last CSC station (ME4), there is no detector behind the LLP decay, so a shower is not seen. At small z, before ME1/1, the decay happens in the thick steel of the yoke, where the shower can be absorbed before it produces enough rechits to form a cluster.

Between the stations, the efficiency is modulated by the steel. Decays just in front of a steel layer develop a shower that reaches the chambers, while decays inside a gas gap or right after a steel layer produce less. Read the station-by-station pattern off the plot and compare it with the steel positions in drawCSCz. I did not run the notebook, so the exact values are not quoted.


# Ex: Plot the 2D histogram of LLP and cluster positions

- Why does the efficiency drops off at the two ends of the muon detectors?
- How does the efficiency varies between the muon stations? Do you understand why?
- Make a 2D histogram to confirm your understanding!

In [ ]:
hz= hist.Hist("Events",
              hist.Bin("z_llp", "LLP decay Z[cm]", 35, 400, 1075),
              hist.Bin("z_cls", "Cluster Z[cm]", 35, 400, 1075)
             )
# Event mask: events that have a matched cluster (numerator events)
ev_numer = ak.sum(numer_sel, axis=1) > 0

# Get the LLP decay position in CSC, for the numerator events
llp_z = ak.sum(abs(signal.gLLP_decay_vertex_z[denom_sel]), axis=1)[ev_numer]

# Plot the position of the matched cluster in z
# - Get the cluster Z position
# - Apply the numerator selection you computed above
# NOTE: the cluster Z branch name is an assumption; confirm it with signal.fields (cell 48)
cls_z = ak.sum(abs(signal.cscRechitClusterZ[numer_sel]), axis=1)[ev_numer]

# Fill the 2D
hz.fill(z_llp=llp_z, z_cls = cls_z)


**Explanation.** The 2D histogram puts the LLP decay Z on one axis and the matched cluster Z on the other. Entries lie near the diagonal when the cluster sits at the decay point. Off-diagonal spread shows that the shower develops over several cm of steel, so the cluster Z is shifted downstream of the decay. ev_numer keeps the two arrays the same length.

In [ ]:
# Plot the 2D histogram
hist.plot2d(hz,xaxis="z_llp")

**Discussion 3.3 (answer, part 3).** Question, verbatim: "Make a 2D histogram to confirm your understanding!"

The 2D histogram shows the cluster Z against the LLP decay Z for matched events. A band along the diagonal means clusters form near the decay. Its width and the shift of the band show how far the shower develops before the cluster is formed. The pattern supports the efficiency interpretation above: decays at positions that do not give a developed shower in the chambers appear as a gap in the band.
